In [1]:
# Parameters
BATCH_MODE = "true"


# SC-22-Solana-Anchor - Solana avec Anchor

[<< Move & Sui](SC-21-Move-Sui.ipynb) | [Cross-Chain >>](../06-Real-World/../06-Real-World/SC-23-Cross-Chain.ipynb)

---

## Objectifs d'apprentissage

1. Comprendre l'architecture **Solana** (accounts, programs)
2. Utiliser le framework **Anchor** (Rust)
3. Creer des **PDAs** (Program Derived Addresses)
4. Implementer des **CPIs** (Cross-Program Invocations)

### Prerequis

- [SC-1](../01-Solidity-Foundation/SC-3-Solidity-Basics.ipynb) a [SC-4](../01-Solidity-Foundation/SC-6-Errors-Events.ipynb) completes (concepts)
- Notions de base en Rust (variables, ownership, structs)
- Solana CLI et Anchor installe (voir setup dans le notebook)

### Duree estimee : 55 minutes

---

## 1. Architecture Solana

Solana utilise un modele different d'Ethereum.

In [2]:
# Comparaison Ethereum vs Solana
print("""
ETHEREUM vs SOLANA

| Concept       | Ethereum              | Solana                  |
|--------------|----------------------|-------------------------|
| Execution    | EVM                   | BPF (Berkeley Packet Filter) |
| State        | Storage in contract  | Accounts (separates)    |
| Langage      | Solidity              | Rust (via Anchor)       |
| Adressage    | Nonces sequentielles  | Hash d'accounts         |
| Fees         | Gas (variable)        | Compute units + rent    |
| TPS          | ~15-30                | ~65,000                 |

COMPOSANTS SOLANA:
- Program    : Code executable (stateless)
- Account    : Stockage de donnees (state)
- PDA        : Program Derived Address
- CPI        : Cross-Program Invocation
- Token      : SPL Token (equivalent ERC-20/721)
""")


ETHEREUM vs SOLANA

| Concept       | Ethereum              | Solana                  |
|--------------|----------------------|-------------------------|
| Execution    | EVM                   | BPF (Berkeley Packet Filter) |
| State        | Storage in contract  | Accounts (separates)    |
| Langage      | Solidity              | Rust (via Anchor)       |
| Adressage    | Nonces sequentielles  | Hash d'accounts         |
| Fees         | Gas (variable)        | Compute units + rent    |
| TPS          | ~15-30                | ~65,000                 |

COMPOSANTS SOLANA:
- Program    : Code executable (stateless)
- Account    : Stockage de donnees (state)
- PDA        : Program Derived Address
- CPI        : Cross-Program Invocation
- Token      : SPL Token (equivalent ERC-20/721)



---

## 2. Programme Anchor Simple

In [3]:
# Counter en Anchor
ANCHOR_COUNTER = '''
// Fichier: programs/counter/src/lib.rs

use anchor_lang::prelude::*;

declare_id!("Counter11111111111111111111111111111111111");

#[program]
pub mod counter {
    use super::*;

    pub fn initialize(ctx: Context<Initialize>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count = 0;
        counter.authority = ctx.accounts.authority.key();
        Ok(())
    }

    pub fn increment(ctx: Context<Increment>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count += 1;
        Ok(())
    }

    pub fn decrement(ctx: Context<Decrement>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        require!(counter.count > 0, CounterError::Underflow);
        counter.count -= 1;
        Ok(())
    }
}

#[derive(Accounts)]
pub struct Initialize<'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Counter::INIT_SPACE,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    #[account(mut)]
    pub authority: Signer<'info>,
    pub system_program: Program<'info, System>,
}

#[derive(Accounts)]
pub struct Increment<'info> {
    #[account(
        mut,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    pub authority: Signer<'info>,
}

#[derive(Accounts)]
pub struct Decrement<'info> {
    #[account(
        mut,
        seeds = [b"counter", authority.key().as_ref()],
        bump
    )]
    pub counter: Account<'info, Counter>,
    pub authority: Signer<'info>,
}

#[account]
#[derive(InitSpace)]
pub struct Counter {
    pub authority: Pubkey,
    pub count: u64,
}

#[error_code]
pub enum CounterError {
    #[msg("Counter cannot go below zero")]
    Underflow,
}
'''

print("Programme Counter Anchor:")
print(ANCHOR_COUNTER)

Programme Counter Anchor:

// Fichier: programs/counter/src/lib.rs

use anchor_lang::prelude::*;

declare_id!("Counter11111111111111111111111111111111111");

#[program]
pub mod counter {
    use super::*;

    pub fn initialize(ctx: Context<Initialize>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count = 0;
        counter.authority = ctx.accounts.authority.key();
        Ok(())
    }

    pub fn increment(ctx: Context<Increment>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        counter.count += 1;
        Ok(())
    }

    pub fn decrement(ctx: Context<Decrement>) -> Result<()> {
        let counter = &mut ctx.accounts.counter;
        require!(counter.count > 0, CounterError::Underflow);
        counter.count -= 1;
        Ok(())
    }
}

#[derive(Accounts)]
pub struct Initialize<'info> {
    #[account(
        init,
        payer = authority,
        space = 8 + Counter::INIT_SPACE,
        seeds = [b"counter", authority.k

### Interpretation : Architecture Solana vs Ethereum

Ce tableau compare les differents modeles d'execution et de stockage entre Ethereum et Solana.

| Concept | Ethereum | Solana | Impact pratique |
|---------|----------|---------|-----------------|
| **Execution** | EVM (bytecode stack-based) | BPF (Berkeley Packet Filter, compile depuis Rust/C++) | Solana est plus rapide (JIT, native compilation) |
| **State** | Stocke dans le contrat (storage) | Comptes separes du programme | Solana : plus de parallelisme, pas de "global state" |
| **Langage** | Solidity (type, haut niveau) | Rust (systeme, verifie) | Rust est plus strict (ownership) mais plus performant |
| **Adressage** | Nonces sequentielles (contract creation count) | Hash des seeds du PDA | Solana : addresses deterministes, previsibles |
| **Fees** | Gas (variable selon complexite) | Compute units + rent exemption | Solana : couts predictibles, rent pour stockage durable |
| **TPS** | ~15-30 (Layer 1) | ~65,000 (theorique) | Solana : haut debit, sharding natif |

**Points cles** :
1. **Programs stateless** : Le code est separe des donnees (comptes). Le programme ne contient pas d'etat.
2. **Accounts model** : Chaque compte est une entite separee avec son propre solde (Lamports) et ses donnees.
3. **Rent exemption** : Les comptes doivent avoir un solde minimum pour exister (rent). Au-dessus de ce seuil, pas de frais de stockage.
4. **Parallelisme** : Solana peut executer plusieurs transactions en parallele si elles touchent des comptes differents.

**Implications pour les developpeurs** :
- Sur Ethereum : on deploye un contrat avec tout l'etat (mappings, arrays)
- Sur Solana : on deploye un programme (code) et on cree des comptes PDAs pour chaque utilisateur/entite
- Le modele Solana permet plus de parallelisme mais demande plus de planification (structure des comptes)

> **Note technique** : BPF (Berkeley Packet Filter) est un format bytecode securise qui s'execute dans le noyau Linux. Solana l'utilise pour la securite et la performance (JIT compilation vers machine code).


---

## 3. PDAs (Program Derived Addresses)

Les PDAs sont des adresses derivees deterministes.

In [4]:
# PDA explanation
print("""
PDA (Program Derived Address)

PROPRIETES:
- Adresse derivee du program_id et de seeds
- Pas de cle privee (programme seulement)
- Deterministe: meme seeds = meme PDA
- Bump: [b"seed", bump] pour trouver le PDA

GENERATION:
seeds = [b"counter", authority.key().as_ref()]
(pda, bump) = Pubkey::find_program_address(&seeds, program_id)

UTILISATION:
1. Stockage de donnees utilisateur
2. Escrows
3. Token accounts
4. Metadata

EXEMPLE DANS ANCHOR:
#[account(
    seeds = [b"counter", authority.key().as_ref()],
    bump
)]
pub counter: Account<'info, Counter>,
""")


PDA (Program Derived Address)

PROPRIETES:
- Adresse derivee du program_id et de seeds
- Pas de cle privee (programme seulement)
- Deterministe: meme seeds = meme PDA
- Bump: [b"seed", bump] pour trouver le PDA

GENERATION:
seeds = [b"counter", authority.key().as_ref()]
(pda, bump) = Pubkey::find_program_address(&seeds, program_id)

UTILISATION:
1. Stockage de donnees utilisateur
2. Escrows
3. Token accounts
4. Metadata

EXEMPLE DANS ANCHOR:
#[account(
    seeds = [b"counter", authority.key().as_ref()],
    bump
)]
pub counter: Account<'info, Counter>,



### Interpretation : Programme Counter Anchor

Ce code presente un programme de compteur simple avec les concepts fondamentaux d'Anchor.

| Composant | Role | Details |
|-----------|------|---------|
| `#[program]` | Module de fonctions publiques | Points d'entree du programme (appelables par les clients) |
| `Context<Initialize>` | Validation des comptes | Anchor verifie que tous les comptes passes sont valides |
| `#[account(init)]` | Creation de compte | Alloue un nouveau compte PDA avec `init` + `payer` + `space` + `seeds` |
| `#[account(mut)]` | Compte modifiable | Le compte peut etre modifie pendant la transaction |
| `#[error_code]` | Erreurs custom | Erreurs specifiques au programme (ici: `Underflow`) |

**Points cles** :
1. `initialize` cree un PDA `counter` avec seeds `[b"counter", authority.key().as_ref()]` → 1 counter par utilisateur
2. `space = 8 + Counter::INIT_SPACE` : 8 octets de discriminant Anchor + taille de la struct `Counter` (32 + 8 = 40)
3. `increment` et `decrement` utilisent les memes seeds pour retrouver le PDA du counter
4. `require!(counter.count > 0, CounterError::Underflow)` : protection contre underflow (custom error)

**Comparaison Solidity vs Anchor** :
- **Solidity** : `mapping(address => uint256) public counters;` (stocke dans le contrat)
- **Anchor** : Chaque utilisateur a son propre compte PDA `Counter` (separe du programme)

**Avantages du modele Solana** :
- Parallelelisme : chaque utilisateur modifie son propre compte (pas de contention)
- Rent exemption : l'utilisateur paie pour son propre compte (pas le deployeur)
- Flexibilite : les comptes peuvent etre fermes pour recuperer la rent

> **Note technique** : Le discriminant Anchor (8 octets) est un hash unique qui identifie le type de compte. Il permet a Anchor de verifier qu'un compte passe est bien du type attendu.


---

## 4. CPI (Cross-Program Invocation)

In [5]:
# CPI example
CPI_EXAMPLE = '''
// CPI vers System Program pour transfer SOL
use anchor_lang::system_program;

pub fn transfer_sol(ctx: Context<TransferSol>, amount: u64) -> Result<()> {
    let cpi_context = CpiContext::new(
        ctx.accounts.system_program.to_account_info(),
        system_program::Transfer {
            from: ctx.accounts.from.to_account_info(),
            to: ctx.accounts.to.to_account_info(),
        }
    );
    system_program::transfer(cpi_context, amount)
}

#[derive(Accounts)]
pub struct TransferSol<'info> {
    #[account(mut)]
    pub from: Signer<'info>,
    #[account(mut)]
    pub to: AccountInfo<'info>,
    pub system_program: Program<'info, System>,
}

// CPI vers Token Program
use anchor_spl::token::{self, Transfer as TokenTransfer};

pub fn transfer_token(
    ctx: Context<TransferToken>,
    amount: u64
) -> Result<()> {
    let cpi_accounts = TokenTransfer {
        from: ctx.accounts.from.to_account_info(),
        to: ctx.accounts.to.to_account_info(),
        authority: ctx.accounts.authority.to_account_info(),
    };
    let cpi_program = ctx.accounts.token_program.to_account_info();
    token::transfer(CpiContext::new(cpi_program, cpi_accounts), amount)
}
'''

print("CPI Examples:")
print(CPI_EXAMPLE)

CPI Examples:

// CPI vers System Program pour transfer SOL
use anchor_lang::system_program;

pub fn transfer_sol(ctx: Context<TransferSol>, amount: u64) -> Result<()> {
    let cpi_context = CpiContext::new(
        ctx.accounts.system_program.to_account_info(),
        system_program::Transfer {
            from: ctx.accounts.from.to_account_info(),
            to: ctx.accounts.to.to_account_info(),
        }
    );
    system_program::transfer(cpi_context, amount)
}

#[derive(Accounts)]
pub struct TransferSol<'info> {
    #[account(mut)]
    pub from: Signer<'info>,
    #[account(mut)]
    pub to: AccountInfo<'info>,
    pub system_program: Program<'info, System>,
}

// CPI vers Token Program
use anchor_spl::token::{self, Transfer as TokenTransfer};

pub fn transfer_token(
    ctx: Context<TransferToken>,
    amount: u64
) -> Result<()> {
    let cpi_accounts = TokenTransfer {
        from: ctx.accounts.from.to_account_info(),
        to: ctx.accounts.to.to_account_info(),
        a

### Interpretation : PDAs (Program Derived Addresses)

Les PDAs sont une innovation unique de Solana : des adresses sans cle privee, controlees par des programmes.

| Propriete | Description | Impact |
|-----------|-------------|--------|
| **Pas de cle privee** | Le PDA est derive de `program_id + seeds` | Seul le programme peut signer pour le PDA |
| **Deterministe** | Memes seeds = meme PDA (toujours) | Pas de collision, pas de generation aleatoire |
| **Bump canonique** | Trouve le premier `bump` (1-255) qui donne une adresse hors courbe ed25519 | Garantit que seul le programme peut signer |

**Points cles** :
1. `seeds = [b"counter", authority.key().as_ref()]` : le PDA est unique par utilisateur
2. Le `bump` est la valeur qui fait que l'adresse est "hors courbe" (pas de cle privee correspondante)
3. Anchor calcule automatiquement le bump avec `#[account(seeds = [...], bump)]`
4. Les PDAs sont utilises pour : comptes de donnees utilisateur, vaults, escrows, metadata NFT

**Cas d'usage typiques** :
- **Compte utilisateur** : `seeds = [b"user", user_pubkey.as_ref()]` → 1 compte par utilisateur
- **Vault d'escrow** : `seeds = [b"vault", escrow_id.as_ref()]` → vault unique par escrow
- **Metadata NFT** : `seeds = [b"metadata", mint.as_ref()]` → metadata unique par mint

**Generation manuelle** (Rust) :
```rust
let (pda, bump) = Pubkey::find_program_address(
    &[b"counter", authority.key().as_ref()],
    program_id
);
```

> **Note technique** : Le "bump" est entre 1 et 255. Si 255 ne suffit pas, il n'y a pas de PDA valide. Anchor utilise le "bump canonique" (le premier qui marche) pour garantir l'unicite.


---

## 5. Commandes Anchor CLI

In [6]:
# Anchor CLI commands
print("""
INSTALLATION:
cargo install --git https://github.com/coral-xyz/anchor avm --locked --force
avm install latest
avm use latest

CREATION PROJET:
anchor init my_program
cd my_program

STRUCTURE:
my_program/
|-- Anchor.toml          # Configuration
|-- Cargo.toml
|-- programs/
|   `-- my_program/
|       |-- Cargo.toml
|       `-- src/
|           `-- lib.rs   # Code principal
|-- tests/
|   `-- my_program.ts    # Tests
`-- app/                 # Frontend

COMMANDES:
anchor build              # Compiler
anchor test               # Executer les tests
anchor deploy             # Deploier
anchor run <script>       # Executer un script

CLUSTER:
anchor test --skip-local-validator  # Avec validator externe
anchor deploy --provider.cluster devnet
""")


INSTALLATION:
cargo install --git https://github.com/coral-xyz/anchor avm --locked --force
avm install latest
avm use latest

CREATION PROJET:
anchor init my_program
cd my_program

STRUCTURE:
my_program/
|-- Anchor.toml          # Configuration
|-- Cargo.toml
|-- programs/
|   `-- my_program/
|       |-- Cargo.toml
|       `-- src/
|           `-- lib.rs   # Code principal
|-- tests/
|   `-- my_program.ts    # Tests
`-- app/                 # Frontend

COMMANDES:
anchor build              # Compiler
anchor test               # Executer les tests
anchor deploy             # Deploier
anchor run <script>       # Executer un script

CLUSTER:
anchor test --skip-local-validator  # Avec validator externe
anchor deploy --provider.cluster devnet



### Interpretation : Cross-Program Invocations (CPI)

Les CPIs permettent a un programme Solana d'appeler un autre programme (equivalent des `delegatecall` en Solidity).

| Type de CPI | Programme cible | Operation |
|-------------|----------------|-----------|
| `system_program::transfer` | System Program | Transferts de SOL native |
| `token::transfer` | SPL Token Program | Transferts de tokens SPL |
| Metaplex CPI | Metaplex | Creation NFT, metadata |
| Custom CPI | Autres programmes utilisateur | Composition de fonctionnalites |

**Points cles** :
1. `CpiContext::new` cree un contexte CPI standard (signataire = l'appelant)
2. `CpiContext::new_with_signer` permet a un PDA de signer (cas du vault escrow)
3. Tous les comptes necessaires doivent etre passes dans la struct `Accounts` (Anchor verifie la coherence)
4. Les CPIs sont payables par l'appelant (compute units + rent exemption)

**Difference EVM vs Solana** :
- EVM : `delegatecall` execute le code d'un autre contrat DANS le contexte du contrat appelant
- Solana : CPI execute le code d'un autre programme AVEC ses propres comptes (contexte separe)

**Pattern CPI classique** :
1. Definir les comptes necessaires dans la struct `#[derive(Accounts)]`
2. Creer le `CpiContext` avec les bons comptes et autorites
3. Appeler la fonction du programme cible (ex: `token::transfer`)

> **Note technique** : Les CPIs sont COMPOSES sur Solana. Un programme A peut appeler B, qui appelle C, etc. La limite de profondeur est de 4 appels chaines.


---

## 6. Exemples guidés

In [7]:
# Exercice: Token Escrow
EXERCISE_ESCROW = '''
#[program]
pub mod escrow {
    use super::*;

    pub fn make(
        ctx: Context<Make>,
        seed: u64,
        receive_amount: u64,
    ) -> Result<()> {
        let escrow = &mut ctx.accounts.escrow;
        escrow.seed = seed;
        escrow.initializer = ctx.accounts.initializer.key();
        escrow.mint_a = ctx.accounts.mint_a.key();
        escrow.mint_b = ctx.accounts.mint_b.key();
        escrow.receive_amount = receive_amount;

        // Transfer tokens to vault
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.deposit.to_account_info(),
                    to: ctx.accounts.vault.to_account_info(),
                    authority: ctx.accounts.initializer.to_account_info(),
                },
            ),
            ctx.accounts.deposit.amount,
        )
    }

    pub fn take(ctx: Context<Take>) -> Result<()> {
        // Transfer B to initializer
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.depositor_token_b.to_account_info(),
                    to: ctx.accounts.initializer_token_b.to_account_info(),
                    authority: ctx.accounts.depositor.to_account_info(),
                },
            ),
            ctx.accounts.escrow.receive_amount,
        )?;

        // Transfer A to depositor
        token::transfer(
            CpiContext::new_with_signer(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.vault.to_account_info(),
                    to: ctx.accounts.depositor_token_a.to_account_info(),
                    authority: ctx.accounts.escrow.to_account_info(),
                },
                &[&[
                    b"escrow",
                    ctx.accounts.maker.key().as_ref(),
                    &ctx.accounts.escrow.seed.to_le_bytes()[..],
                    &[ctx.bumps.escrow],
                ]],
            ),
            ctx.accounts.vault.amount,
        )
    }
}
'''
print("Exercice Escrow:")
print(EXERCISE_ESCROW)

Exercice Escrow:

#[program]
pub mod escrow {
    use super::*;

    pub fn make(
        ctx: Context<Make>,
        seed: u64,
        receive_amount: u64,
    ) -> Result<()> {
        let escrow = &mut ctx.accounts.escrow;
        escrow.seed = seed;
        escrow.initializer = ctx.accounts.initializer.key();
        escrow.mint_a = ctx.accounts.mint_a.key();
        escrow.mint_b = ctx.accounts.mint_b.key();
        escrow.receive_amount = receive_amount;

        // Transfer tokens to vault
        token::transfer(
            CpiContext::new(
                ctx.accounts.token_program.to_account_info(),
                token::Transfer {
                    from: ctx.accounts.deposit.to_account_info(),
                    to: ctx.accounts.vault.to_account_info(),
                    authority: ctx.accounts.initializer.to_account_info(),
                },
            ),
            ctx.accounts.deposit.amount,
        )
    }

    pub fn take(ctx: Context<Take>) -> Result<()> {


### Interpretation : Commandes Anchor CLI

Ces commandes couvrent le cycle de vie complet de developpement d'un programme Solana avec Anchor.

| Commande | Action | Usage typique |
|----------|--------|---------------|
| `anchor init my_program` | Creer un nouveau projet | Demarrage d'un nouveau programme |
| `anchor build` | Compiler le programme Rust | Apres modification du code |
| `anchor test` | Executer les tests TypeScript | Verification avant deploiement |
| `anchor deploy` | Deploier sur le cluster configure | Mise en production (devnet/mainnet) |
| `anchor run <script>` | Executer un script TypeScript | Operations one-off (mint, init, etc.) |

**Points cles** :
1. La structure de projet separe le code (`programs/`) des tests (`tests/`) et du frontend (`app/`)
2. `Anchor.toml` contient la configuration du cluster (localnet, devnet, mainnet) et les features du programme
3. Les tests sont ecrits en TypeScript (avec `@coral-xyz/anchor`) et interagissent avec le programme comme un client le ferait
4. `--skip-local-validator` permet de se connecter a un validator externe (devnet Solana publique) au lieu de lancer un validator local

**Workflow typique** :
```
anchor init mon_program    # Creer le projet
anchor build                # Compiler (verifier la syntaxe Rust)
anchor test                # Executer les tests (verifier la logique)
anchor deploy               # Deploier sur devnet
anchor run scripts/mint.ts  # Executer des operations d'initialisation
```

> **Note technique** : Anchor utilise `avm` (Anchor Version Manager) pour gerer les versions multiples. `avm install latest` installe la derniere version, `avm use latest` l'active.


---

## 7. Resume

| Concept | Description |
|---------|-------------|
| Program | Code executable stateless |
| Account | Stockage de donnees |
| PDA | Adresse derivee du programme |
| CPI | Appel inter-programmes |
| Anchor | Framework Rust pour Solana |

---

**Notebook suivant** : [SC-23-Cross-Chain](../06-Real-World/SC-23-Cross-Chain.ipynb)

---

[<< Move & Sui](SC-21-Move-Sui.ipynb) | [Cross-Chain >>](../06-Real-World/../06-Real-World/SC-23-Cross-Chain.ipynb)

### Interpretation : Token Escrow avec CPI

Cet exercice presente un escrow atomique (echange de tokens sans tiers de confiance) utilisant des CPIs.

| Fonction | Operation | CPI utilise |
|----------|-----------|-------------|
| `make` | Initialise l'escrow et transfere Token A vers vault | `token::transfer` (initializer → vault) |
| `take` | Echange atomique : Token B → initializer, Token A → depositor | 2x `token::transfer` (depositor → initializer, vault → depositor) |

**Points cles** :
1. `make` cree un compte escrow avec seeds `[b"escrow", maker.key(), seed]` (PDA deterministe)
2. Le vault est un compte token PDA controle par l'escrow (pas de cle privee)
3. `take` utilise `CpiContext::new_with_signer` avec les seeds de l'escrow PDA pour autoriser le transfert depuis le vault
4. L'echange est atomique : soit les 2 transferts reussissent, soit aucun (revert si l'un echoue)

**Securite** :
- Le seed `u64` permet a un utilisateur d'avoir plusieurs escrows simultanes
- Le bump `ctx.bumps.escrow` est inclus dans la signature CPI pour prouver l'autorite du PDA
- L'ordre des transferts est important : d'abord le token B du depositor, puis le token A du vault (evite certains attaques)

> **Note technique** : Sur Solana, les escrows sont des PDAs (pas de contrats intelligents avec etat). Le PDA signe les transferts via `new_with_signer`, ce qui est impossible sur EVM.


### Exercice 3 : Simuler un CPI (Cross-Program Invocation) Solana

Simulez en Python un appel cross-program Solana ou un programme A appelle
une fonction d'un programme B (equivalent d'un delegatecall Ethereum).

**Objectifs** :
1. Modeliser Programme A (caller) et Programme B (callee)
2. Implementer `cpi_call(program_a, program_b, accounts, data)`
3. Verifier que le callee execute dans le contexte du caller

**Indice** : CPI = un programme Solana peut appeler une instruction d'un autre programme.


In [8]:
# Exercice 3 : Simuler un CPI Solana (Cross-Program Invocation)
# TODO etudiant : simuler un appel cross-programme
# Etape 1 : Definir ProgramA avec une fonction qui appelle ProgramB
# Etape 2 : Definir ProgramB avec une fonction de traitement
# Etape 3 : Simuler cpi_call(programs, accounts, instruction_data)
class SolanaProgram:
    def __init__(self, name):
        self.name = name
        self.accounts = {}

    def execute(self, instruction, accounts, data):
        # TODO etudiant : executer l'instruction
        return {"status": "ok"}  # TODO etudiant

# program_a = SolanaProgram("token_program")
# program_b = SolanaProgram("system_program")
# result = program_a.execute("transfer", {"from": "alice", "to": "bob"}, {"amount": 100})
print("Exercice a completer")


Exercice a completer


### Exercice 2 : Contrat Anchor de compteur avec increment et reset

Ecrivez un programme Anchor (Rust simplifie) qui gere un compteur avec increment
et reset, en utilisant le pattern Anchor `#[derive(Accounts)]`.

**Objectifs** :
1. Definir le compte `Counter` avec `count: u64`
2. Implementer `increment(ctx)` et `reset(ctx)`
3. Ecrire la structure de validation `#[derive(Accounts)]`

**Indice** : Anchor utilise `ctx.accounts.counter` pour acceder au compte deserialise.


In [9]:
# Exercice 2 : Compteur Anchor (Rust simplifie)
# TODO etudiant : ecrire un programme Anchor Counter en pseudo-Rust
# Etape 1 : Definir #[account] struct Counter { count: u64 }
# Etape 2 : Implementer increment(ctx) { ctx.accounts.counter.count += 1; }
# Etape 3 : Implementer reset(ctx) { ctx.accounts.counter.count = 0; }
anchor_counter = """
// TODO etudiant
// use anchor_lang::prelude::*;
//
// #[program]
// pub mod counter {
//     use super::*;
//     pub fn increment(ctx: Context<Increment>) -> Result<()> { ... }
//     pub fn reset(ctx: Context<Reset>) -> Result<()> { ... }
// }
//
// #[account]
// pub struct Counter { count: u64 }
//
// #[derive(Accounts)]
// pub struct Increment<'info> { ... }
"""
print("Exercice a completer")


Exercice a completer


## Resume et perspectives

Ce notebook a permis de decourir l'ecosysteme Solana et le framework Anchor comme alternative a l'ecosysteme Ethereum/Solidity. Nous avons compare les architectures : modele de comptes stateless et execution BPF haute performance (~65 000 TPS theoriques) chez Solana versus stockage dans le contrat et EVM chez Ethereum. Nous avons implemente un programme Counter en Rust avec Anchor, explorant les annotations `#[program]`, `#[account]` et `#[derive(Accounts)]` qui abstraient la complexite du modele de comptes. Nous avons etudie les PDAs (Program Derived Addresses) pour le stockage deterministe et securise de donnees utilisateur, les CPIs (Cross-Program Invocations) pour la composition de programmes, et implémente un escrow atomique de tokens SPL combinant `CpiContext::new` et `CpiContext::new_with_signer` pour les transfers signes par PDA.

La difference philosophique majeure entre Solana et Ethereum reside dans la separation stricte entre code (programmes stateless) et donnees (comptes independants). Ce modele enable le parallelisme natif des transactions et elimine la contention globale, mais exige une planification plus rigoureuse de la structure des comptes. Les PDAs remplacent les mappings Solidity par des adresses deterministes sans cle privee, offrant une securite inherente contre les acces non autorises. Le framework Anchor simplifie considerablement le developpement en Rust tout en preservant les garanties de securite du type system.

Le notebook suivant, [SC-23-Cross-Chain](../06-Real-World/../06-Real-World/SC-23-Cross-Chain.ipynb), aborde l'interoperabilite entre blockchains et les protocols de communication cross-chain comme Chainlink CCIP.